# 0.1 前沿优化器 (Advanced Optimizers)

> 🕐 预估学习时间：40分钟

AdamW 仍是 LLM 默认优化器，但 2024–2026 出现一批面向大规模训练的新选择：Lion、Sophia、Muon、SOAP 等，在收敛速度、显存与稳定性上各有取舍。

本节涵盖：
- AdamW 基线回顾
- Lion / Sophia 的更新规则
- Muon（正交化动量）简化实现
- SOAP / Shampoo 族直觉
- 选型建议


## 1. AdamW 基线

AdamW = 一阶动量 + 二阶自适应 + **解耦权重衰减**。大模型训练的事实标准，但二阶统计量占用额外显存，且对某些矩阵结构参数未必最优。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import copy

torch.manual_seed(42)


class TinyLM(nn.Module):
    def __init__(self, vocab=64, d=32):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.fc = nn.Linear(d, vocab)

    def forward(self, x):
        return self.fc(self.embed(x).mean(1))


def make_batch(n=64, t=8, vocab=64):
    x = torch.randint(0, vocab, (n, t))
    y = torch.randint(0, vocab, (n,))
    return x, y


def train_steps(model, opt, steps=40):
    losses = []
    for _ in range(steps):
        x, y = make_batch()
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses


base = TinyLM()
m = copy.deepcopy(base)
opt = torch.optim.AdamW(m.parameters(), lr=3e-3, weight_decay=0.01)
adamw_losses = train_steps(m, opt)
print('=== AdamW Baseline ===')
print(f'loss start={adamw_losses[0]:.4f} mid={adamw_losses[20]:.4f} end={adamw_losses[-1]:.4f}')
print(f'Key: AdamW remains the default; new optimizers must beat it on wall-clock or stability.')


## 2. Lion：符号动量，省显存

Lion 用动量的符号更新参数，几乎不存二阶统计，更新幅度更离散。适合大批次、需要省优化器状态的场景。

$$\theta \leftarrow \theta - \eta \cdot \mathrm{sign}(m) - \eta\lambda\theta$$


In [ ]:
class Lion(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        super().__init__(params, dict(lr=lr, betas=betas, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            beta1, beta2 = group['betas']
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['exp_avg'] = torch.zeros_like(p)
                exp_avg = state['exp_avg']
                update = exp_avg * beta1 + grad * (1 - beta1)
                p.add_(update.sign(), alpha=-group['lr'])
                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['lr'] * group['weight_decay'])
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)


m = copy.deepcopy(base)
lion_losses = train_steps(m, Lion(m.parameters(), lr=1e-3, weight_decay=0.01))
print('=== Lion ===')
print(f'loss start={lion_losses[0]:.4f} end={lion_losses[-1]:.4f}')
print(f'optimizer state bytes ~ 1x params (momentum only), vs AdamW ~2x')
print(f'Key: Lion trades second-moment adaptivity for lower memory and sign-based updates.')


## 3. Sophia：对角 Hessian 近似

Sophia 用对角二阶信息缩放梯度（可用 Gauss-Newton / Hutchinson 估计），并做裁剪防止过大步长。对 LLM 预训练有报告显示可减少步数。


In [ ]:
class SophiaG(torch.optim.Optimizer):
    '''Educational Sophia-G: EMA of grad^2 as diagonal curvature proxy.'''
    def __init__(self, params, lr=1e-3, betas=(0.965, 0.99), rho=0.04, weight_decay=0.01):
        super().__init__(params, dict(lr=lr, betas=betas, rho=rho, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            beta1, beta2 = group['betas']
            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['m'] = torch.zeros_like(p)
                    state['h'] = torch.zeros_like(p)
                m, h = state['m'], state['h']
                m.mul_(beta1).add_(grad, alpha=1 - beta1)
                h.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                if group['weight_decay'] != 0:
                    p.mul_(1 - group['lr'] * group['weight_decay'])
                # update = clip(m / max(h, eps), +/- rho)
                denom = h.clamp_min(1e-12)
                update = (m / denom).clamp(-group['rho'], group['rho'])
                p.add_(update, alpha=-group['lr'])


m = copy.deepcopy(base)
sophia_losses = train_steps(m, SophiaG(m.parameters(), lr=3e-3))
print('=== Sophia-G (toy) ===')
print(f'loss start={sophia_losses[0]:.4f} end={sophia_losses[-1]:.4f}')
print(f'Key: Sophia scales steps by curvature proxy and clips; useful when AdamW needs many tokens.')


## 4. Muon：对二维参数做正交化动量

Muon 把矩阵参数的动量做 Newton–Schulz 风格正交化，使更新更接近“谱友好”的方向，在中等规模 LLM 实验中表现突出。偏置/一维参数通常仍用 AdamW。


In [ ]:
def newton_schulz_orthogonalize(G, steps=5):
    '''Approximate polar factor / orthogonalize a matrix via Newton-Schulz.'''
    X = G / (G.norm() + 1e-8)
    if G.size(0) > G.size(1):
        X = X.T
        transposed = True
    else:
        transposed = False
    for _ in range(steps):
        A = X @ X.T
        B = A @ X
        X = 1.5 * X - 0.5 * B
    if transposed:
        X = X.T
    return X


class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True):
        super().__init__(params, dict(lr=lr, momentum=momentum, nesterov=nesterov))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['buf'] = torch.zeros_like(g)
                buf = state['buf']
                buf.mul_(group['momentum']).add_(g)
                update = g.add(buf, alpha=group['momentum']) if group['nesterov'] else buf
                if update.ndim == 2:
                    update = newton_schulz_orthogonalize(update)
                else:
                    update = update / (update.norm() + 1e-8)
                p.add_(update, alpha=-group['lr'])


m = copy.deepcopy(base)
# Muon for 2D weights, AdamW for embeddings/biases (simplified: all Muon)
muon_losses = train_steps(m, Muon(m.parameters(), lr=0.05), steps=40)
print('=== Muon ===')
print(f'loss start={muon_losses[0]:.4f} end={muon_losses[-1]:.4f}')
W = m.fc.weight.detach()
print(f'fc weight singular values (top3): {torch.linalg.svdvals(W)[:3].tolist()}')
print(f'Key: Muon orthogonalizes matrix momentum; pair with AdamW for 1D params in practice.')


## 5. SOAP / Shampoo 族直觉与选型

| 优化器 | 核心想法 | 显存 | 典型场景 |
|--------|---------|------|---------|
| AdamW | 对角二阶自适应 | 中 | 通用默认 |
| Lion | 符号动量 | 低 | 省状态、大批次 |
| Sophia | 对角曲率 + 裁剪 | 中 | 减少预训练步数 |
| Muon | 矩阵动量正交化 | 低-中 | 隐藏层矩阵为主 |
| SOAP/Shampoo | 分层预条件 | 高 | 研究/中等规模 |

**实践建议**：先 AdamW 拉通；当 token 预算或稳定性成为瓶颈再 A/B 新优化器；始终固定数据顺序与种子对比。


In [ ]:
print('=== Optimizer Comparison (toy CE) ===')
print(f'{"opt":<10} {"start":>8} {"end":>8} {"delta":>8}')
for name, losses in [('AdamW', adamw_losses), ('Lion', lion_losses),
                     ('Sophia', sophia_losses), ('Muon', muon_losses)]:
    print(f'{name:<10} {losses[0]:>8.4f} {losses[-1]:>8.4f} {losses[0]-losses[-1]:>8.4f}')
print(f'\nKey: Toy losses are not ranking evidence; use tokens-to-target on real runs.')


## 课后思考题

1. 为什么 Muon 主要作用于二维权重矩阵，而 embedding/LM head 仍常用 AdamW？
2. Sophia 的 rho 裁剪过大或过小分别会怎样？
3. 在 FSDP/ZeRO-3 下换优化器时，通信与状态分片要注意什么？
4. 如何设计公平的优化器对比实验（数据、调度、数值精度）？

---
> 本节涵盖了0.1 前沿优化器的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
